In [ ]:
%load_ext cash
%cash_on
%cash_badge print
%cash_debug on

In [ ]:
# ── Cell 2: Imports ──
import pandas as pd
import numpy as np
import os
import time
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Cell 3: Generate synthetic BTS flight data (~2M flights) ──
# Mimics real BTS On-Time Performance data structure
_t0 = time.time()
_data_dir = os.path.join(os.getcwd(), 'data', 'flights')
os.makedirs(_data_dir, exist_ok=True)
_csv_path = os.path.join(_data_dir, 'flights_2024.csv')

if not os.path.exists(_csv_path):
    print('Generating synthetic flight data...')
    _rng = np.random.default_rng(42)
    _n = 2_000_000
    
    # Carrier codes and hub airports
    _carriers = ['AA', 'DL', 'UA', 'WN', 'B6', 'AS', 'NK', 'F9', 'G4', 'HA']
    _airports = ['ATL', 'DFW', 'DEN', 'ORD', 'LAX', 'CLT', 'MCO', 'LAS',
                 'PHX', 'MIA', 'SEA', 'JFK', 'EWR', 'SFO', 'IAH', 'BOS',
                 'FLL', 'MSP', 'DTW', 'PHL', 'LGA', 'BWI', 'SLC', 'DCA',
                 'SAN', 'MDW', 'TPA', 'HNL', 'PDX', 'STL']
    
    # Generate vectorized data
    _months = _rng.integers(1, 13, size=_n)
    _days = _rng.integers(1, 29, size=_n)  # simplified: 1-28 for all months
    _dow = _rng.integers(1, 8, size=_n)  # 1=Mon, 7=Sun
    _dep_hours = _rng.choice(range(5, 24), size=_n, p=np.array([0.02,0.08,0.10,0.10,0.08,0.07,0.06,0.05,0.04,0.04,0.04,0.04,0.05,0.05,0.06,0.05,0.04,0.02,0.01]) / np.array([0.02,0.08,0.10,0.10,0.08,0.07,0.06,0.05,0.04,0.04,0.04,0.04,0.05,0.05,0.06,0.05,0.04,0.02,0.01]).sum())
    _dep_mins = _rng.integers(0, 60, size=_n)
    _crs_dep = _dep_hours * 100 + _dep_mins  # HHMM format
    
    _carrier_idx = _rng.integers(0, len(_carriers), size=_n)
    _origin_idx = _rng.integers(0, len(_airports), size=_n)
    # Ensure dest != origin
    _dest_offset = _rng.integers(1, len(_airports), size=_n)
    _dest_idx = (_origin_idx + _dest_offset) % len(_airports)
    
    # Delay model: most flights on time, some delayed
    _base_delay = _rng.normal(0, 15, size=_n)  # baseline noise
    _weather_factor = (_months <= 2) | (_months >= 11)  # winter = more delays
    _rush_factor = (_dep_hours >= 16) & (_dep_hours <= 20)  # evening rush
    _dep_delay = _base_delay + _weather_factor * _rng.exponential(10, _n) + _rush_factor * _rng.exponential(5, _n)
    _dep_delay = np.round(_dep_delay).astype(int)
    
    # Arrival delay correlated with departure delay
    _arr_delay = _dep_delay + _rng.normal(-2, 8, size=_n)  # airlines try to make up time
    _arr_delay = np.round(_arr_delay).astype(int)
    
    # Distance (miles)
    _distance = _rng.choice([250, 500, 750, 1000, 1500, 2000, 2500, 3000, 4000, 5000], size=_n,
                           p=[0.15, 0.20, 0.15, 0.15, 0.12, 0.08, 0.06, 0.04, 0.03, 0.02])
    
    # Cancellation (~2%)
    _cancelled = _rng.random(size=_n) < 0.02
    _cancel_codes = np.where(_cancelled, _rng.choice(['A','B','C','D'], size=_n), '')
    
    # Elapsed time = distance / 500mph * 60min + taxi time
    _air_time = np.round(_distance / 500 * 60 + _rng.normal(0, 10, _n)).astype(int)
    _air_time = np.maximum(_air_time, 30)
    _taxi_out = _rng.integers(5, 35, size=_n)
    _taxi_in = _rng.integers(3, 20, size=_n)
    _elapsed = _air_time + _taxi_out + _taxi_in
    
    # Delay causes (only for delayed flights)
    _is_delayed = _arr_delay > 15
    _carrier_delay = np.where(_is_delayed, _rng.exponential(10, _n).astype(int), 0)
    _weather_delay = np.where(_is_delayed & _weather_factor, _rng.exponential(15, _n).astype(int), 0)
    _nas_delay = np.where(_is_delayed, _rng.exponential(8, _n).astype(int), 0)
    _security_delay = np.where(_is_delayed & (_rng.random(_n) < 0.05), _rng.integers(5, 60, _n), 0)
    _late_aircraft = np.where(_is_delayed, _rng.exponential(12, _n).astype(int), 0)
    
    _flight_nums = _rng.integers(100, 9999, size=_n)
    
    _df_gen = pd.DataFrame({
        'Year': 2024,
        'Month': _months,
        'DayofMonth': _days,
        'DayOfWeek': _dow,
        'FlightDate': pd.to_datetime({'year': 2024, 'month': _months, 'day': _days}),
        'Reporting_Airline': np.array(_carriers)[_carrier_idx],
        'Flight_Number_Reporting_Airline': _flight_nums,
        'Origin': np.array(_airports)[_origin_idx],
        'Dest': np.array(_airports)[_dest_idx],
        'CRSDepTime': _crs_dep,
        'DepDelay': _dep_delay,
        'ArrDelay': _arr_delay,
        'Cancelled': _cancelled.astype(int),
        'CancellationCode': _cancel_codes,
        'AirTime': _air_time,
        'Distance': _distance,
        'TaxiOut': _taxi_out,
        'TaxiIn': _taxi_in,
        'ActualElapsedTime': _elapsed,
        'CarrierDelay': _carrier_delay,
        'WeatherDelay': _weather_delay,
        'NASDelay': _nas_delay,
        'SecurityDelay': _security_delay,
        'LateAircraftDelay': _late_aircraft,
    })
    
    _df_gen.to_csv(_csv_path, index=False)
    print(f'Generated {len(_df_gen):,} flights → {os.path.getsize(_csv_path)/1e6:.0f} MB')
    del _df_gen
else:
    print(f'Flight data already exists: {os.path.getsize(_csv_path)/1e6:.0f} MB')

print(f'Time: {time.time()-_t0:.1f}s')

In [ ]:
# ── Cell 4: Load flight data ──
_t0 = time.time()
_csv_path = os.path.join(os.getcwd(), 'data', 'flights', 'flights_2024.csv')
flights = pd.read_csv(_csv_path, parse_dates=['FlightDate'])
print(f'Loaded {len(flights):,} flights, {flights.memory_usage(deep=True).sum()/1e6:.0f} MB RAM')
print(f'Columns: {list(flights.columns)}')
print(f'Date range: {flights["FlightDate"].min()} to {flights["FlightDate"].max()}')
print(f'Carriers: {sorted(flights["Reporting_Airline"].unique())}')
print(f'Time: {time.time()-_t0:.1f}s')

In [ ]:
# ── Cell 5: Feature engineering for delay analysis ──
_t0 = time.time()

# Extract time features
flights['DepHour'] = flights['CRSDepTime'] // 100
flights['IsWeekend'] = flights['DayOfWeek'].isin([6, 7]).astype(int)
flights['IsWinter'] = flights['Month'].isin([1, 2, 11, 12]).astype(int)
flights['IsSummer'] = flights['Month'].isin([6, 7, 8]).astype(int)

# Delay categories
flights['IsDelayed'] = (flights['ArrDelay'] > 15).astype(int)
flights['IsSevereDelay'] = (flights['ArrDelay'] > 60).astype(int)
flights['DepDelayBin'] = pd.cut(flights['DepDelay'], 
                                 bins=[-np.inf, -5, 0, 15, 30, 60, np.inf],
                                 labels=['Early', 'OnTime', 'Slight', 'Moderate', 'Major', 'Severe'])

# Distance categories
flights['DistanceBin'] = pd.cut(flights['Distance'],
                                 bins=[0, 500, 1000, 2000, 6000],
                                 labels=['Short', 'Medium', 'Long', 'CrossCountry'])

# Route identifier
flights['Route'] = flights['Origin'] + '-' + flights['Dest']

# Active (non-cancelled) flights
active_flights = flights[flights['Cancelled'] == 0].copy()

print(f'Feature engineering complete: {len(flights.columns)} columns')
print(f'Active flights: {len(active_flights):,} ({len(active_flights)/len(flights)*100:.1f}%)')
print(f'Delayed flights: {int(active_flights["IsDelayed"].sum()):,} ({active_flights["IsDelayed"].mean()*100:.1f}%)')
print(f'Time: {time.time()-_t0:.1f}s')

In [ ]:
# ── Cell 6: Carrier performance analysis ──
_t0 = time.time()

carrier_stats = active_flights.groupby('Reporting_Airline').agg(
    total_flights=('ArrDelay', 'count'),
    avg_dep_delay=('DepDelay', 'mean'),
    avg_arr_delay=('ArrDelay', 'mean'),
    median_arr_delay=('ArrDelay', 'median'),
    pct_delayed=('IsDelayed', 'mean'),
    pct_severe=('IsSevereDelay', 'mean'),
    avg_distance=('Distance', 'mean'),
    avg_air_time=('AirTime', 'mean'),
    cancel_rate=('Cancelled', 'count'),  # placeholder, recalc below
).round(2)

# Actual cancellation rate from full dataset
_cancel_rates = flights.groupby('Reporting_Airline')['Cancelled'].mean().round(4)
carrier_stats['cancel_rate'] = _cancel_rates

# On-time performance score (OTP) = % arriving within 15 min of schedule
carrier_stats['otp_score'] = (1 - carrier_stats['pct_delayed']).round(4)

carrier_stats = carrier_stats.sort_values('otp_score', ascending=False)
print('=== Carrier Performance Rankings ===')
print(carrier_stats[['total_flights', 'avg_arr_delay', 'pct_delayed', 'otp_score', 'cancel_rate']].to_string())
print(f'\nBest on-time: {carrier_stats.index[0]} ({carrier_stats["otp_score"].iloc[0]*100:.1f}%)')
print(f'Worst on-time: {carrier_stats.index[-1]} ({carrier_stats["otp_score"].iloc[-1]*100:.1f}%)')
print(f'Time: {time.time()-_t0:.1f}s')

In [ ]:
# ── Cell 7: Route analysis — busiest routes and delay hotspots ──
_t0 = time.time()

route_stats = active_flights.groupby('Route').agg(
    flights=('ArrDelay', 'count'),
    avg_delay=('ArrDelay', 'mean'),
    pct_delayed=('IsDelayed', 'mean'),
    avg_distance=('Distance', 'mean'),
).round(2)

# Filter to routes with enough flights for statistical significance
route_stats = route_stats[route_stats['flights'] >= 100]

# Top 20 busiest routes
busiest_routes = route_stats.nlargest(20, 'flights')
print('=== Top 20 Busiest Routes ===')
print(busiest_routes.to_string())

# Most delayed routes (min 200 flights)
_sig_routes = route_stats[route_stats['flights'] >= 200]
worst_routes = _sig_routes.nlargest(15, 'pct_delayed')
print(f'\n=== Top 15 Most Delayed Routes (min 200 flights) ===')
print(worst_routes.to_string())

# Airport delay stats
origin_delays = active_flights.groupby('Origin').agg(
    departures=('DepDelay', 'count'),
    avg_dep_delay=('DepDelay', 'mean'),
    pct_delayed=('IsDelayed', 'mean'),
).round(2).sort_values('pct_delayed', ascending=False)

print(f'\n=== Airport Departure Delay Rankings ===')
print(origin_delays.head(15).to_string())
print(f'\nTotal unique routes: {len(route_stats):,}')
print(f'Time: {time.time()-_t0:.1f}s')

In [ ]:
# ── Cell 8: Temporal patterns — delay by month, day, hour ──
_t0 = time.time()

# Monthly delay pattern
monthly_stats = active_flights.groupby('Month').agg(
    flights=('ArrDelay', 'count'),
    avg_delay=('ArrDelay', 'mean'),
    pct_delayed=('IsDelayed', 'mean'),
    avg_dep_delay=('DepDelay', 'mean'),
).round(2)

print('=== Monthly Delay Patterns ===')
print(monthly_stats.to_string())

# Day-of-week pattern
_dow_names = {1:'Mon',2:'Tue',3:'Wed',4:'Thu',5:'Fri',6:'Sat',7:'Sun'}
dow_stats = active_flights.groupby('DayOfWeek').agg(
    flights=('ArrDelay', 'count'),
    avg_delay=('ArrDelay', 'mean'),
    pct_delayed=('IsDelayed', 'mean'),
).round(2)
dow_stats.index = dow_stats.index.map(_dow_names)

print(f'\n=== Day-of-Week Patterns ===')
print(dow_stats.to_string())

# Hourly pattern
hourly_stats = active_flights.groupby('DepHour').agg(
    flights=('ArrDelay', 'count'),
    avg_delay=('ArrDelay', 'mean'),
    pct_delayed=('IsDelayed', 'mean'),
).round(2)

print(f'\n=== Hourly Departure Delay Patterns ===')
print(hourly_stats.to_string())

# Find worst time to fly
_worst_hour = int(hourly_stats['pct_delayed'].idxmax())
_worst_month = int(monthly_stats['pct_delayed'].idxmax())
print(f'\nWorst hour: {_worst_hour}:00 ({hourly_stats.loc[_worst_hour, "pct_delayed"]*100:.1f}% delayed)')
print(f'Worst month: {_worst_month} ({monthly_stats.loc[_worst_month, "pct_delayed"]*100:.1f}% delayed)')
print(f'Time: {time.time()-_t0:.1f}s')

In [ ]:
# ── Cell 9: Delay cause decomposition ──
_t0 = time.time()

delay_cols = ['CarrierDelay', 'WeatherDelay', 'NASDelay', 'SecurityDelay', 'LateAircraftDelay']
_delayed_flights = active_flights[active_flights['IsDelayed'] == 1]

# Overall delay cause breakdown
delay_totals = _delayed_flights[delay_cols].sum()
delay_pcts = (delay_totals / delay_totals.sum() * 100).round(1)

print('=== Delay Cause Breakdown (% of total delay minutes) ===')
for _col, _pct in delay_pcts.items():
    print(f'  {_col:25s}: {_pct:5.1f}%  ({int(delay_totals[_col]):>12,} minutes)')

# Delay causes by carrier
carrier_delay_causes = _delayed_flights.groupby('Reporting_Airline')[delay_cols].mean().round(1)
print(f'\n=== Average Delay Minutes by Cause per Carrier ===')
print(carrier_delay_causes.to_string())

# Delay causes by season
_delayed_flights_s = _delayed_flights.copy()
_season_map = {1:'Winter',2:'Winter',3:'Spring',4:'Spring',5:'Spring',
               6:'Summer',7:'Summer',8:'Summer',9:'Fall',10:'Fall',11:'Fall',12:'Winter'}
_delayed_flights_s['Season'] = _delayed_flights_s['Month'].map(_season_map)
season_delays = _delayed_flights_s.groupby('Season')[delay_cols].mean().round(1)
print(f'\n=== Average Delay Minutes by Cause per Season ===')
print(season_delays.to_string())

print(f'\nTotal delayed flights analyzed: {len(_delayed_flights):,}')
print(f'Time: {time.time()-_t0:.1f}s')

In [ ]:
# ── Cell 10: ML - Delay prediction model ──
_t0 = time.time()
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score

# Prepare features for delay prediction
_model_cols = ['Month', 'DayOfWeek', 'DepHour', 'Distance', 'IsWeekend', 'IsWinter', 'IsSummer']

# Add carrier as numeric encoding
_carrier_map = {c: i for i, c in enumerate(sorted(active_flights['Reporting_Airline'].unique()))}
active_flights['CarrierCode'] = active_flights['Reporting_Airline'].map(_carrier_map)
_model_cols.append('CarrierCode')

# Sample for faster training (stratified)
_sample = active_flights.sample(n=200_000, random_state=42)
X = _sample[_model_cols].values
y = _sample['IsDelayed'].values

print(f'Training set: {len(X):,} flights, {y.mean()*100:.1f}% delayed')
print(f'Features: {_model_cols}')

# Train GBM classifier with cross-validation
model = GradientBoostingClassifier(
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1,
    subsample=0.8,
    random_state=42
)

cv_scores = cross_val_score(model, X, y, cv=5, scoring='roc_auc', n_jobs=-1)
print(f'\n5-Fold CV ROC AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Per-fold: {[f"{s:.4f}" for s in cv_scores]}')

# Fit final model
model.fit(X, y)

# Feature importance
importances = pd.Series(model.feature_importances_, index=_model_cols).sort_values(ascending=False)
print(f'\n=== Feature Importance ===')
for _feat, _imp in importances.items():
    print(f'  {_feat:20s}: {_imp:.4f}')

print(f'\nTime: {time.time()-_t0:.1f}s')

In [ ]:
# ── Cell 11: Summary statistics and final report ──
_t0 = time.time()

# Top 5 most reliable carriers
_reliable = carrier_stats.head(5)
# Top 5 most unreliable
_unreliable = carrier_stats.tail(5)

# Overall summary
total_flights = len(flights)
total_active = len(active_flights)
total_cancelled = int(flights['Cancelled'].sum())
overall_delay_rate = float(active_flights['IsDelayed'].mean())
avg_delay = float(active_flights['ArrDelay'].mean())
median_delay = float(active_flights['ArrDelay'].median())

print('=' * 60)
print('US FLIGHTS ON-TIME PERFORMANCE REPORT - 2024')
print('=' * 60)
print(f'Total flights:        {total_flights:>12,}')
print(f'Active flights:       {total_active:>12,}')
print(f'Cancelled flights:    {total_cancelled:>12,} ({total_cancelled/total_flights*100:.1f}%)')
print(f'Overall delay rate:   {overall_delay_rate*100:>11.1f}%')
print(f'Avg arrival delay:    {avg_delay:>11.1f} min')
print(f'Median arrival delay: {median_delay:>11.1f} min')
print(f'Unique routes:        {len(route_stats):>12,}')
print(f'Unique airports:      {flights["Origin"].nunique():>12,}')
print(f'ML model AUC:         {cv_scores.mean():>11.4f}')
print(f'\nData size in memory:  {flights.memory_usage(deep=True).sum()/1e6:.0f} MB')
print('=' * 60)
print(f'Time: {time.time()-_t0:.1f}s')